In [2]:
"""
Jupyter-friendly Tic-Tac-Toe Q-Learning implementation.

✔ No argparse
✔ Clean functions you can call directly in a notebook
✔ Symmetry reduction
✔ Training, evaluation, and play vs human

Usage inside notebook:

from tic_tac_toe_q_learning_jupyter import *
agent = QLearningAgent()
stats = train(agent, 50000)
evaluate(agent, 2000)
human_vs_agent(agent)

"""

import random
import pickle
from collections import defaultdict

# =============================
# Board utilities
# =============================
WIN_LINES = [
    (0,1,2),(3,4,5),(6,7,8),
    (0,3,6),(1,4,7),(2,5,8),
    (0,4,8),(2,4,6)
]

def empty_board():
    return (0,)*9

def available_actions(state):
    return [i for i,v in enumerate(state) if v==0]

def apply_action(state, action, player):
    s = list(state)
    s[action] = player
    return tuple(s)

def check_winner(state):
    for a,b,c in WIN_LINES:
        s = state[a] + state[b] + state[c]
        if s == 3:  return 1
        if s == -3: return -1
    if all(v != 0 for v in state):
        return 0
    return None

# =============================
# Symmetry transforms (D4 group)
# =============================
rot90 = [6,3,0,7,4,1,8,5,2]
flip  = [2,1,0,5,4,3,8,7,6]

def perm_comp(a,b):
    return [a[i] for i in b]

r1 = rot90
r2 = perm_comp(rot90, rot90)
r3 = perm_comp(r2, rot90)
IDENT = list(range(9))

TRANSFORMS = [
    IDENT, r1, r2, r3,
    flip,
    perm_comp(flip, r1),
    perm_comp(flip, r2),
    perm_comp(flip, r3)
]

def transform_state(state, perm):
    return tuple(state[i] for i in perm)

def canonical_state(state):
    return min(transform_state(state, p) for p in TRANSFORMS)

# =============================
# Q-Learning Agent
# =============================
class QLearningAgent:
    def __init__(self, alpha=0.5, gamma=0.9, epsilon=0.1, use_symmetry=True):
        self.Q = defaultdict(float)
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.use_symmetry = use_symmetry

    def state_key(self, state):
        return canonical_state(state) if self.use_symmetry else state

    def get_q(self, state, action):
        return self.Q[(self.state_key(state), action)]

    def best_action(self, state):
        acts = available_actions(state)
        if not acts:
            return None, 0
        qs = [(a, self.get_q(state, a)) for a in acts]
        maxq = max(v for _, v in qs)
        best = [a for a,v in qs if v==maxq]
        return random.choice(best), maxq

    def choose_action(self, state, training=True):
        acts = available_actions(state)
        if not acts:
            return None
        if training and random.random() < self.epsilon:
            return random.choice(acts)
        a,_ = self.best_action(state)
        return a

    def update(self, state, action, reward, next_state):
        key = (self.state_key(state), action)
        old = self.Q[key]
        if next_state is None:
            target = reward
        else:
            acts = available_actions(next_state)
            if acts:
                max_next = max(self.Q[(self.state_key(next_state), a)] for a in acts)
            else:
                max_next = 0
            target = reward + self.gamma * max_next
        self.Q[key] = old + self.alpha * (target - old)

    def save(self, path):
        with open(path, "wb") as f:
            pickle.dump(dict(self.Q), f)

    def load(self, path):
        with open(path, "rb") as f:
            data = pickle.load(f)
        self.Q = defaultdict(float, data)

# =============================
# Episode simulation
# =============================

def run_episode(agent, opponent="random", training=True):
    state = empty_board()
    player = 1
    transitions = []

    while True:
        if player == 1:
            action = agent.choose_action(state, training)
        else:
            acts = available_actions(state)
            if opponent == "random":
                action = random.choice(acts)
            elif opponent == "copy_agent":
                action = agent.choose_action(state, training)

        next_state = apply_action(state, action, player)
        res = check_winner(next_state)

        if res is not None:
            if res == 1:
                r_agent = 1
            elif res == -1:
                r_agent = -1
            else:
                r_agent = 0.5

            # store transitions
            transitions.append((state, action, r_agent if player==1 else 0, None, player))
            break

        transitions.append((state, action, 0, next_state, player))
        state = next_state
        player *= -1

    # update agent moves
    for s,a,r,ns,p in transitions:
        if p == 1:
            agent.update(s, a, r, ns)

    return res

# =============================
# Training
# =============================

def train(agent, episodes=10000, opponent="random", decay=True):
    stats = {1:0, -1:0, 0:0}
    for ep in range(1, episodes+1):
        res = run_episode(agent, opponent, training=True)
        stats[res] += 1
        if decay:
            agent.epsilon = max(0.01, agent.epsilon * 0.9999)
    return stats

# =============================
# Evaluation
# =============================

def evaluate(agent, games=1000, opponent="random"):
    old_eps = agent.epsilon
    agent.epsilon = 0
    stats = {1:0, -1:0, 0:0}
    for _ in range(games):
        res = run_episode(agent, opponent, training=False)
        stats[res] += 1
    agent.epsilon = old_eps
    return stats

# =============================
# Human play
# =============================

def print_board(state):
    s = {1:"X", -1:"O", 0:" "}
    print("\n".join([
        f"{s[state[0]]}|{s[state[1]]}|{s[state[2]]}",
        f"{s[state[3]]}|{s[state[4]]}|{s[state[5]]}",
        f"{s[state[6]]}|{s[state[7]]}|{s[state[8]]}"
    ]))
    print()


def human_vs_agent(agent):
    print("You are O. Agent is X and goes first.")
    state = empty_board()
    player = 1

    while True:
        print_board(state)

        if player == 1:
            a = agent.choose_action(state, training=False)
            print(f"Agent plays {a}")
            state = apply_action(state, a, 1)
        else:
            acts = available_actions(state)
            print("Available moves:", acts)
            m = None
            while m not in acts:
                try:
                    m = int(input("Your move (0-8): "))
                except:
                    pass
            state = apply_action(state, m, -1)

        res = check_winner(state)
        if res is not None:
            print_board(state)
            if res == 1:
                print("Agent wins!")
            elif res == -1:
                print("You win!")
            else:
                print("Draw.")
            break
        player *= -1
